# Cool Method - Character Networks (who appears with whom)

An optional starter for **story projects**.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset (mount Drive; falls back to a local folder) ---
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
%pip install -q networkx

In [ ]:
import re, itertools
from collections import Counter
import networkx as nx
import matplotlib.pyplot as plt
print("imports ok")

In [ ]:
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode:", SMOKE)

## Your story, and the cast

In class this is *your* fiction corpus (a novel, a fanfic, a screenplay).

In [ ]:
text = """
Alice met Bob in the garden, and Bob told Alice about Cara.
Later Cara found Alice by the river while Bob waited at home.
Dan arrived; Dan and Bob argued, and Cara calmed Dan down.
Alice and Cara left together, but Bob and Dan stayed behind.
Alice thought of Cara often; Bob rarely thought of anyone but Dan.
"""
characters = ["Alice", "Bob", "Cara", "Dan"]
print("cast:", characters)

## Build the co-occurrence graph

Slide a window of `WINDOW` words across the text; every pair of characters that co-occurs inside
a window gets its tie strengthened.

In [ ]:
WINDOW = 15
tokens = re.findall(r"[A-Za-z]+", text)
pair_counts = Counter()
for i in range(len(tokens)):
    window = set(tokens[i:i+WINDOW])
    present = [c for c in characters if c in window]
    for a, b in itertools.combinations(sorted(present), 2):
        pair_counts[(a, b)] += 1

G = nx.Graph()
G.add_nodes_from(characters)
for (a, b), w in pair_counts.items():
    G.add_edge(a, b, weight=w)
print("ties:", dict(pair_counts))

## Who holds the story together?

**Degree** = how many others a character shares scenes with.

In [ ]:
deg = nx.degree_centrality(G)
btw = nx.betweenness_centrality(G, weight="weight")
print("Most central by degree:     ", sorted(deg, key=deg.get, reverse=True))
print("Most central by betweenness:", sorted(btw, key=btw.get, reverse=True))

plt.figure(figsize=(5,4))
pos = nx.spring_layout(G, seed=1)
sizes = [3000*deg[n] for n in G.nodes()]
nx.draw_networkx(G, pos, node_size=sizes, node_color="#cdd1e0", font_size=9)
widths = [G[u][v]["weight"] for u,v in G.edges()]
nx.draw_networkx_edges(G, pos, width=widths, alpha=0.5)
plt.axis("off"); plt.title(f"Character network (window={WINDOW} words)"); plt.tight_layout(); plt.show()

## You made a character network

Now change `WINDOW` to 5, then 40, and re-run.

**Sketch:** run this on your own corpus, report the most central character at two window sizes,
and say which you believe and why.